# 02. 모델 · 입력 크기 비교 학습

여러 조합을 **짧게(20바퀴)** 학습해서 비교하고, 제일 좋은 조합만 **길게(60바퀴)** 다시 학습해요.

| 순서 | MODEL | IMGSZ | EPOCHS | 목적 | 예상 시간(T4) |
| --- | --- | --- | --- | --- | --- |
| 1 | yolov8n | 960 | 20 | 기준 | 약 45분 |
| 2 | yolov8n | 640 | 20 | 입력 크기 비교 | 약 20분 |
| 3 | yolov8n | 1280 | 20 | 입력 크기 비교 | 약 1시간 30분 |
| 4 | yolov8s | 960 | 20 | 모델 크기 비교 | 약 1시간 30분 |
| 5 | yolo11n | 960 | 20 | 모델 세대 비교 | 약 50분 |
| 6 | (1~5 중 최고) | (최고) | 60 | 최종 베이스라인 | 2~4시간 |

- **5번 칸의 `MODEL`, `IMGSZ`, `EPOCHS` 세 줄만** 바꿔서 한 번씩 실행하면 돼요.
- 비교는 **dev 셋(쪽지시험)** 점수로만 해요. 시험지(eval)로 고르면 커닝이에요.
- 결과는 **Google Drive**에 저장돼요. 접속이 끊겨도 같은 설정으로 다시 실행하면 이어서 학습해요.
- 무료 Colab은 하루 GPU 사용 시간이 정해져 있어서, 하루에 2~3개씩 나눠서 하는 게 안전해요.
- 시간은 대략적인 값이에요.
- `CUDA out of memory` 에러가 나면: 그때까지 한 바퀴들은 저장돼 있어요. 세션 다시 시작 → 1~4번 → 5번을 같은 설정으로 다시 실행하면 이어서 해요.

## 1. GPU 확인

`Tesla T4` 가 나오면 OK. 에러가 나면 런타임 → 런타임 유형 변경 → **T4 GPU** 로 바꿔 주세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive 연결

권한 허용 창이 뜨면 허용해 주세요. 학습 결과가 `내 드라이브/embeded_ondeviceAI/runs` 에 저장돼요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS = '/content/drive/MyDrive/embeded_ondeviceAI/runs'
os.makedirs(RUNS, exist_ok=True)

## 3. 데이터 준비

01 노트북과 같은 작업이에요. 이번 접속에서 01을 이미 실행했다면 금방 끝나요.

In [ ]:
%%bash
set -e   # 중간에 실패하면 바로 멈추고 어디서 실패했는지 보여 줌
git clone -q https://github.com/gyu-young-04/embeded_ondeviceAI.git /content/embeded_ondeviceAI 2>/dev/null || git -C /content/embeded_ondeviceAI pull -q
mkdir -p /content/kitti && cd /content/kitti
for f in data_object_label_2.zip data_object_image_2.zip; do
  wget -c -q https://s3.eu-central-1.amazonaws.com/avg-kitti/$f
  unzip -q -n $f 'training/*'
done
[ "$(ls training/image_2 | wc -l)" = 7481 ] || { echo '사진이 7481장이 아니에요. /content/kitti 를 지우고 다시 실행하세요.'; exit 1; }
cd /content/embeded_ondeviceAI
[ -f /content/kitti_yolo/kitti.yaml ] || python scripts/prepare_kitti.py --kitti /content/kitti --out /content/kitti_yolo
wc -l /content/kitti_yolo/train.txt /content/kitti_yolo/dev.txt

## 4. Ultralytics 설치

출력되는 버전을 보고서의 '실험 환경'에 적어 두세요.

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)

## 5. 학습 ⭐ 여기 세 줄만 바꿔서 실행

- 위 표의 순서대로 `MODEL`, `IMGSZ`, `EPOCHS` 를 바꾸고 ▶ 를 눌러요.
- 끊기면 1~4번을 다시 실행한 뒤 이 칸을 **같은 설정으로** 실행하면 이어서 학습해요.
- 화면에 나오는 mAP50 / mAP50-95 는 COCO 방식이라 **대회 점수(KITTI AP40)와 달라요.** 비교용으로만 봐요.

In [ ]:
MODEL = 'yolov8n'   # yolov8n / yolov8s / yolo11n
IMGSZ = 960         # 640 / 960 / 1280
EPOCHS = 20         # 비교할 때 20, 최종 학습은 60

from pathlib import Path
import torch
from ultralytics import YOLO

NAME = f'{MODEL}_{IMGSZ}_e{EPOCHS}'
# 큰 모델(s)이나 큰 사진(1280)은 GPU 메모리(15GB)가 부족해서 한 번에 16장씩
BATCH = 16 if IMGSZ >= 1280 or MODEL.endswith('s') else 32
last = Path(RUNS) / NAME / 'weights' / 'last.pt'

def finished(ckpt):
    # Ultralytics 는 학습이 끝나면 체크포인트의 epoch 를 -1 로 바꿔 둔다
    return torch.load(ckpt, map_location='cpu', weights_only=False).get('epoch', 0) == -1

if last.exists() and finished(last):
    print('이미 끝난 실험이에요:', NAME)
elif last.exists():
    print('이어서 학습:', last)
    YOLO(str(last)).train(resume=True, batch=BATCH)
else:
    YOLO(f'{MODEL}.pt').train(
        data='/content/kitti_yolo/kitti.yaml',
        imgsz=IMGSZ,
        epochs=EPOCHS,
        patience=20,      # 20바퀴 동안 나아지지 않으면 일찍 멈춤
        batch=BATCH,
        cache='ram',      # 메모리가 부족하면 자동으로 꺼져요
        seed=0,
        deterministic=True,
        project=RUNS,
        name=NAME,
        exist_ok=True,
    )

## 6. 실험 비교표

Drive에 저장된 모든 실험을 표로 모아요. **`dev mAP50-95` 가 높을수록 정확**해요.
표는 `runs/comparison.csv` 로도 저장돼요. 속도(latency) 비교는 다음 노트북에서 따로 재요.

In [ ]:
import pandas as pd
import torch
from pathlib import Path

rows = []
for run in sorted(Path(RUNS).iterdir()):
    csv = run / 'results.csv'
    if not csv.exists():
        continue
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    best = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
    last = run / 'weights' / 'last.pt'
    done = last.exists() and torch.load(last, map_location='cpu', weights_only=False).get('epoch', 0) == -1
    rows.append({
        '실험': run.name,
        '진행 바퀴': len(df),
        'dev mAP50': round(best['metrics/mAP50(B)'], 3),
        'dev mAP50-95': round(best['metrics/mAP50-95(B)'], 3),
        '학습 시간(분)': round(df['time'].iloc[-1] / 60) if 'time' in df else None,
        '상태': '완료' if done else '진행 중',
    })

table = pd.DataFrame(rows).sort_values('dev mAP50-95', ascending=False)
table.to_csv(Path(RUNS) / 'comparison.csv', index=False)
table

## 7. 클래스별 점수 보기 (선택)

아래 `NAME` 에 보고 싶은 실험 이름을 넣으면 Car / Pedestrian / Cyclist 별 점수를 보여 줘요.

In [ ]:
NAME = 'yolov8n_960_e20'

from IPython.display import Image, display
run = Path(RUNS) / NAME
display(Image(str(run / 'results.png'), width=1000))
imgsz = int(NAME.split('_')[1])
metrics = YOLO(str(run / 'weights' / 'best.pt')).val(data='/content/kitti_yolo/kitti.yaml', imgsz=imgsz, split='val', batch=16)
for i, n in metrics.names.items():
    print(f'{n:<11} mAP50 {metrics.box.ap50[i]:.3f}  mAP50-95 {metrics.box.ap[i]:.3f}')